[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/04_neural_networks/04_neural_networks.ipynb)

# 04. Neural Networks — XOR, Backpropagation, ReLU, Dropout, MNIST

> 원본 강의: [Lec 7–10, 모두를 위한 머신러닝과 딥러닝](https://hunkim.github.io/ml/) — ML 실용 팁, 딥러닝 기본(XOR), Backpropagation, ReLU/초기화/Dropout

## 이론

### 1) 실용 팁: 학습률, Overfitting, Train/Test 분리
- **학습률(learning rate)**: 너무 크면 발산(overshooting), 너무 작으면 학습이 매우 느립니다.
- **Overfitting**: 학습 데이터에만 과하게 맞춰져 새 데이터에서 성능이 떨어지는 현상. **Regularization**, **Dropout**, 더 많은 데이터로 완화합니다.
- **Train/Test 분리**: 모델이 실제로 일반화되는지 확인하려면 학습에 쓰지 않은 데이터로 평가해야 합니다.

### 2) XOR 문제
XOR(배타적 논리합)은 직선 하나로 나눌 수 없는(linearly non-separable) 대표적인 문제입니다. Logistic Regression(퍼셉트론 1개)으로는 풀 수 없고, **은닉층(hidden layer)을 가진 다층 신경망**이 필요합니다 — 이것이 초기 신경망 연구가 침체되었던 이유(1969, Minsky)이자, Backpropagation이 해결한 문제입니다.

### 3) Backpropagation
출력에서 발생한 오차(cost)를 체인 룰(chain rule)을 이용해 역방향으로 전파하며 각 층의 가중치에 대한 그래디언트를 계산하는 알고리즘입니다. 이 덕분에 여러 층을 가진 네트워크도 Gradient Descent로 학습할 수 있습니다.

### 4) ReLU, 가중치 초기화, Dropout
- **Sigmoid의 문제**: 층이 깊어지면 미분값(<1)이 반복 곱해지며 그래디언트가 0에 수렴하는 **Vanishing Gradient** 문제 발생.
- **ReLU**: $f(x) = \max(0, x)$. 양수 구간 기울기가 1이라 깊은 네트워크에서도 그래디언트가 잘 전파됩니다.
- **가중치 초기화**: 0으로 초기화하면 모든 뉴런이 동일하게 학습되어 의미가 없습니다. Xavier/He 초기화처럼 층의 크기에 맞게 무작위 초기화해야 합니다.
- **Dropout**: 학습 시 일부 뉴런을 무작위로 끔으로써 특정 뉴런에 의존하지 않게 하여 Overfitting을 줄이는 규제(regularization) 기법. 여러 서브네트워크를 앙상블하는 효과와 유사합니다.

## 실습 0. Colab 환경 설정 (PyTorch)

이 노트북부터는 신경망을 직접 구성하기 위해 **PyTorch**를 사용합니다. Colab에서 GPU를 쓰려면 상단 메뉴 **런타임 → 런타임 유형 변경 → 하드웨어 가속기 → GPU**로 설정하세요.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)
if IN_COLAB:
    !pip install -q torch torchvision matplotlib

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)
print("device:", device)

## 실습 1. XOR — 단일 퍼셉트론은 실패, 다층 신경망은 성공

In [ ]:
X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]], device=device)
Y = torch.tensor([[0.], [1.], [1.], [0.]], device=device)  # XOR

def train(model, epochs=2000, lr=0.5):
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.BCELoss()
    losses = []
    for epoch in range(epochs):
        optimizer.zero_grad()
        pred = model(X)
        loss = loss_fn(pred, Y)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return losses

In [ ]:
# (1) 단일 퍼셉트론 (Logistic Regression과 동일한 구조) -> XOR을 풀지 못함
single_layer = nn.Sequential(nn.Linear(2, 1), nn.Sigmoid()).to(device)
losses_single = train(single_layer)

with torch.no_grad():
    pred = single_layer(X)
print("단일 퍼셉트론 최종 예측:\n", pred.cpu().numpy().round(3))
print("최종 loss:", losses_single[-1])

In [ ]:
# (2) 은닉층을 가진 다층 신경망(MLP) -> XOR을 성공적으로 학습
mlp = nn.Sequential(
    nn.Linear(2, 8),
    nn.Sigmoid(),
    nn.Linear(8, 1),
    nn.Sigmoid(),
).to(device)
losses_mlp = train(mlp)

with torch.no_grad():
    pred = mlp(X)
print("다층 신경망 최종 예측:\n", pred.cpu().numpy().round(3))
print("최종 loss:", losses_mlp[-1])

In [ ]:
plt.plot(losses_single, label="단일 퍼셉트론 (실패)")
plt.plot(losses_mlp, label="다층 신경망 (성공)")
plt.xlabel("epoch")
plt.ylabel("BCE loss")
plt.title("XOR: 단일 퍼셉트론 vs 다층 신경망")
plt.legend()
plt.show()

## 실습 2. MNIST — ReLU + Dropout으로 손글씨 숫자 분류

`torchvision`으로 MNIST를 내려받고, ReLU 활성화 + Dropout을 적용한 MLP로 학습합니다. Colab GPU를 켰다면 자동으로 사용됩니다.

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([transforms.ToTensor()])

train_ds = datasets.MNIST(root="../../data", train=True, download=True, transform=transform)
test_ds = datasets.MNIST(root="../../data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

len(train_ds), len(test_ds)

In [ ]:
class MnistMLP(nn.Module):
    def __init__(self, dropout_p=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 256),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.net(x)

model = MnistMLP().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb).argmax(dim=1)
            correct += (pred == yb).sum().item()
            total += yb.size(0)
    return correct / total

EPOCHS = 3  # Colab에서 빠르게 체감하도록 3 epoch만; 늘리면 정확도가 더 오릅니다
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * xb.size(0)

    train_loss = running_loss / len(train_ds)
    test_acc = evaluate(model, test_loader)
    print(f"epoch {epoch+1}/{EPOCHS}  train_loss={train_loss:.4f}  test_acc={test_acc:.4f}")

## 정리 & 연습 문제
- 실습 1에서 은닉층의 뉴런 수(8 -> 2)를 줄이면 XOR을 못 풀 수도 있습니다. 직접 바꿔보며 확인해보세요.
- 실습 2에서 `nn.Dropout(dropout_p)`를 제거하고 다시 학습하면 train/test 성능 차이(Overfitting 정도)가 어떻게 달라지는지 비교해보세요.
- `EPOCHS`를 늘리거나 은닉층을 추가해서 정확도를 98% 이상으로 올려보세요 (원본 강의 Lec 10의 목표와 동일).
- 다음 노트북([05_cnn.ipynb](../05_cnn/05_cnn.ipynb))에서는 이미지에 특화된 CNN으로 같은 MNIST를 더 잘 풀어봅니다.


**해설/정답**: [04_neural_networks_solutions.ipynb](04_neural_networks_solutions.ipynb)
